# Inspeção Visual de Peças de Fundição Metálica

Mini-projeto de visão computacional + CNN para classificação binária de peças fundidas
(**OK** vs **Defeituosa**), usando o dataset *Casting Product Image Data for Quality Inspection*.

O fluxo segue seis sprints industriais: ingestão → OpenCV clássico → morfologia →
pipeline Keras → CNN → auditoria de performance.

## Sprint 1 — Configuração e ingestão dos dados

Objetivo: ambiente reprodutível e dataset acessível localmente em `casting_data/`
(`def_front/` e `ok_front/`). Sem seed fixa, qualquer comparação entre execuções
(treino, split, augmentation) fica inválida em auditoria.

In [ ]:
from pathlib import Path
import random
import zipfile

import numpy as np
import tensorflow as tf

# Mesma seed em NumPy, Python e TF: o split 80/20 e o shuffle do dataset
# precisam coincidir entre notebooks do time e entre reexecuções na fábrica.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Determinismo "melhor esforço" no TF (não elimina 100% da variação em GPU,
# mas reduz ruído na comparação de hiperparâmetros).
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"TensorFlow {tf.__version__} | seed={SEED}")

In [ ]:
# Caminhos do projeto
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "casting_data"
DRIVE_FILE_ID = "1NZOjCHDRrpn7PmbFKVqegUP5arfdXHKK"
ZIP_PATH = ROOT / "casting_dataset.zip"


def contar_imagens(pasta: Path) -> int:
    return sum(1 for p in pasta.rglob("*") if p.suffix.lower() in {".jpeg", ".jpg", ".png"})


def dataset_pronto(data_dir: Path) -> bool:
    """Exige as duas classes com pelo menos uma imagem cada (ignora .gitkeep vazio)."""
    d_def, d_ok = data_dir / "def_front", data_dir / "ok_front"
    return (
        d_def.is_dir()
        and d_ok.is_dir()
        and contar_imagens(d_def) > 0
        and contar_imagens(d_ok) > 0
    )


def baixar_dataset_drive(file_id: str, destino_zip: Path) -> None:
    # gdown trata a confirmação de download grande do Drive; requests puro falha com HTML de aviso.
    import gdown

    url = f"https://drive.google.com/uc?id={file_id}"
    print("Baixando dataset do Google Drive...")
    gdown.download(url, str(destino_zip), quiet=False)


def extrair_e_normalizar(zip_path: Path, data_dir: Path) -> None:
    """Extrai o zip e acomoda pastas caso o arquivo venha com um nível extra de diretório."""
    data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)

    # Alguns dumps trazem casting_data/casting_data/... — sobe um nível se necessário.
    nested = data_dir / "casting_data"
    if nested.is_dir() and not dataset_pronto(data_dir):
        for item in nested.iterdir():
            alvo = data_dir / item.name
            if not alvo.exists():
                item.rename(alvo)

    # Caso as classes estejam na raiz do projeto (legado), move para casting_data/
    for classe in ("def_front", "ok_front"):
        origem = ROOT / classe
        destino = data_dir / classe
        if origem.is_dir() and not destino.exists():
            origem.rename(destino)


if dataset_pronto(DATA_DIR):
    print(f"Dataset local encontrado em: {DATA_DIR}")
else:
    if not ZIP_PATH.exists():
        baixar_dataset_drive(DRIVE_FILE_ID, ZIP_PATH)
    extrair_e_normalizar(ZIP_PATH, DATA_DIR)
    if not dataset_pronto(DATA_DIR):
        raise FileNotFoundError(
            "Após o download, não achei def_front/ e ok_front/ em casting_data/. "
            "Confira a estrutura do arquivo do Drive."
        )
    print(f"Dataset preparado em: {DATA_DIR}")

n_def = contar_imagens(DATA_DIR / "def_front")
n_ok = contar_imagens(DATA_DIR / "ok_front")
print(f"Classes | def_front={n_def} | ok_front={n_ok} | total={n_def + n_ok}")
print(
    "Desbalanceamento leve é esperado neste dataset; a métrica accuracy "
    "sozinha pode mascarar falhas na classe minoritária — por isso olhamos as curvas na Sprint 6."
)

## Sprint 2 — EDA clássica (OpenCV: filtros e ruído)

Antes da CNN, inspecionamos o sinal ótico real da esteira: iluminação irregular,
textura do metal e ruído de sensor. Filtros lineares vs. de ordem mostram que tipo
de ruído domina e se ainda precisamos de morfologia na Sprint 3.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def carregar_amostra(pasta: Path, indice: int = 0) -> tuple[np.ndarray, Path]:
    arquivos = sorted(pasta.glob("*.jpeg")) + sorted(pasta.glob("*.jpg"))
    if not arquivos:
        raise FileNotFoundError(f"Nenhuma imagem em {pasta}")
    caminho = arquivos[indice % len(arquivos)]
    # IMREAD_COLOR mantém BGR (padrão OpenCV). Converter cedo demais para RGB
    # só importa na hora de plotar com matplotlib.
    bgr = cv2.imread(str(caminho), cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"Falha ao ler {caminho}")
    return bgr, caminho


bgr_def, path_def = carregar_amostra(DATA_DIR / "def_front", indice=0)
bgr_ok, path_ok = carregar_amostra(DATA_DIR / "ok_front", indice=0)

gray_def = cv2.cvtColor(bgr_def, cv2.COLOR_BGR2GRAY)
gray_ok = cv2.cvtColor(bgr_ok, cv2.COLOR_BGR2GRAY)

# GaussianBlur: convolução com kernel gaussiano — ideal para ruído aproximadamente
# aditivo/gaussiano (sensor). Suaviza também bordas finas; por isso o kernel fica modesto (5x5).
gauss_def = cv2.GaussianBlur(gray_def, (5, 5), sigmaX=1.2)
gauss_ok = cv2.GaussianBlur(gray_ok, (5, 5), sigmaX=1.2)

# medianBlur: cada pixel vira a mediana da vizinhança. Impulsos (sal e pimenta)
# são outliers e somem sem "espalhar" a mancha como a média/gaussiana faria.
# Em fundição, sujeira pontual no sensor/óleo na lente se comporta parecido com salt-pepper.
median_def = cv2.medianBlur(gray_def, 5)
median_ok = cv2.medianBlur(gray_ok, 5)

print("Amostra defeituosa:", path_def.name, gray_def.shape)
print("Amostra OK:", path_ok.name, gray_ok.shape)

In [ ]:
def painel_filtros(titulo: str, gray, gauss, median):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    painéis = [
        (gray, "Original (cinza)"),
        (gauss, "GaussianBlur 5x5"),
        (median, "medianBlur k=5"),
    ]
    for ax, (img, label) in zip(axes, painéis):
        ax.imshow(img, cmap="gray")
        ax.set_title(label)
        ax.axis("off")
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


painel_filtros(f"Defeituosa — {path_def.name}", gray_def, gauss_def, median_def)
painel_filtros(f"OK — {path_ok.name}", gray_ok, gauss_ok, median_ok)

print(
    "Leitura prática: se pontos brancos/pretos isolados somem no median e permanecem "
    "no gaussiano, o ruído dominante é impulsivo. Se o fundo 'enevoa' igual nos dois, "
    "é mais ruído de alta frequência / textura — aí Canny na Sprint 3 precisa de blur prévio."
)